# The master tables — every scored run, decodability BEFORE editability

Reads the `scores.json` files `master_eval.ipynb` writes (same scan: `runs/**`, excluding
`runs/archive/` and `_`-prefixed topics) and renders the cross-run tables, grouped by
topic subdirectory, with an environment column. Deliberately short: every future bespoke
table is a view/subset of these.

**Order is deliberate: decodability first.** An editability number is only interpretable
once the probes demonstrably read the state — an editor writing through a probe that
decodes nothing produces noise, not a negative result. So: **Table 1** decodability
(Probe Skill, the cross-environment axis: 1 = perfect, 0 = trivial baseline; ≡ R² on
regression), with the MLP ≥ linear tripwire count beside it; **Table 1b** discworld
decodability BY COMPONENT (position vs velocity per object — the aggregate is
variance-weighted ~1000:1 toward position, so it hides velocity; Othello's per-tile
equivalent is 64 columns and lives in each run's `scores.json` instead); **Table 2**
editability — each editor's best arm read against the run's unedited EI, never without
its guard (fidelity > 1 on discworld, or li-vs-pre collapsing on Othello, means the
"edit" degraded the model).

In [ ]:
# [1] Collect every scores.json (runs/**; archive/ and _topics excluded), flatten per run.
import json
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

rows = []
for sp in sorted((REPO / "runs").rglob("scores.json")):
    rel = sp.relative_to(REPO / "runs")
    if rel.parts[0] == "archive" or rel.parts[0].startswith("_"):
        continue
    s = json.loads(sp.read_text())
    row = {"topic": rel.parts[0], "run": sp.parent.name, "env": s["env"],
           "arch": s["arch"], "val": s["val_loss"], "eval": s["eval_version"]}
    if s["env"] == "discworld":
        T = s["targets"]["pos"]                      # headline target for the summary
        F = s["targets"].get("full")
        row |= {"skill_lin": max(T["probe_skill_linear"]),
                "skill_mlp": max(T["probe_skill_mlp"]),
                "tripwire": (T["probe_sanity"]["n_violations"]
                             + (F["probe_sanity"]["n_violations"] if F else 0)),
                "unedited_EI": T["unedited"]["edit_index"]}
        if F:  # per-component decodability from the FULL target, at its best point
            best_pt = max(range(len(F["probe_skill_linear"])),
                          key=lambda i: F["probe_skill_linear"][i])
            row["perdim_lin"] = F["probe_perdim_linear"][best_pt]
            row["perdim_mlp"] = F["probe_perdim_mlp"][best_pt]
            row["perdim_point"] = best_pt
        for ed in ("PI", "ND", "GS"):
            b = T["best"].get(ed)
            row[f"{ed}_EI"] = b["edit_index"] if b else None
            row[f"{ed}_guard"] = b["fidelity_ratio"] if b else None
    else:
        sk = s["probe_skill"]
        row |= {"skill_lin": max(sk.get("mine|linear|frame", [float('nan')])),
                "skill_mlp": max(sk.get("state|mlp|frame", [float('nan')])),
                "skill_lin_seq": max(sk.get("mine|linear|sequence", [float('nan')])),
                "skill_mlp_seq": max(sk.get("state|mlp|sequence", [float('nan')])),
                "tripwire": 0,   # classification fits carry no R² tripwire; see probe_stats
                "unedited_EI": s["unedited"]["edit_index_union"],
                "legal_mass": s["gates"]["legal_mass"],
                "ce_excess": s["gates"]["ce"] - s["gates"]["bayes_ce"]}
        for ed in ("PI", "ND", "GS"):
            b = s["best"].get(ed)
            row[f"{ed}_EI"] = b["edit_index_union"] if b else None
            row[f"{ed}_guard"] = b["li_error_vs_pre"] if b else None
    rows.append(row)
print(f"{len(rows)} scored runs")

In [ ]:
# [2] TABLE 1 — DECODABILITY. Read this before the editability table means anything.
#     skill = Probe Skill at the best residual point (≡ R² on regression). Othello shows
#     both split conventions: frame = Li's anchor, seq = this repo's honest convention.
def f(v, spec="+.3f"):
    return "—" if v is None else format(v, spec)

hdr = (f"{'run':<16}{'env':<11}{'arch':<21}{'val':>9}{'skill LIN':>10}{'skill MLP':>10}"
       f"{'LINseq':>8}{'MLPseq':>8}{'tripwire':>9}")
print("TABLE 1 — DECODABILITY (Probe Skill; 1 = perfect, 0 = trivial baseline)")
for topic in sorted({r['topic'] for r in rows}):
    print(f"\n## {topic}\n{hdr}\n{'-' * len(hdr)}")
    for r in sorted((r for r in rows if r['topic'] == topic),
                    key=lambda r: (r['env'], r['run'])):
        print(f"{r['run']:<16}{r['env']:<11}{r['arch']:<21}{r['val']:>9.4f}"
              f"{f(r['skill_lin']):>10}{f(r['skill_mlp']):>10}"
              f"{f(r.get('skill_lin_seq')):>8}{f(r.get('skill_mlp_seq')):>8}"
              f"{r['tripwire']:>9}")
print("\n(discworld skills are the pos target; Othello LIN = mine tiles, MLP = state tiles."
      "\n A nonzero tripwire count means MLP < linear somewhere: treat that run's"
      "\n decodability — and therefore its editability — as untrusted until refit.)")

In [ ]:
# [3] TABLE 1b — DISCWORLD DECODABILITY BY COMPONENT (full target, best residual point).
#     The aggregate skill is variance-weighted ~1000:1 toward position; this is where
#     velocity decodability is actually visible. Two rows per run: LIN and MLP-128.
#     (Othello's per-tile equivalent is 64 columns — it stays in each scores.json.)
COMP = ("o1·x", "o1·y", "o2·x", "o2·y", "o1·vx", "o1·vy", "o2·vx", "o2·vy")

dw = [r for r in rows if r["env"] == "discworld" and "perdim_lin" in r]
if dw:
    hdr = f"{'run':<16}{'probe':<7}{'pt':>3}" + "".join(f"{c:>8}" for c in COMP)
    print("TABLE 1b — DISCWORLD DECODABILITY BY COMPONENT (Probe Skill per dim)")
    print(hdr + "\n" + "-" * len(hdr))
    for r in sorted(dw, key=lambda r: (r["topic"], r["run"])):
        for name, pd in (("LIN", r["perdim_lin"]), ("MLP", r["perdim_mlp"])):
            cells = "".join(f"{v:>+8.3f}" for v in pd[:len(COMP)])
            print(f"{r['run']:<16}{name:<7}{r['perdim_point']:>3}{cells}")
    print("\n(components: object · axis; positions then velocities, in sim units.)")
else:
    print("no discworld runs with per-component decodability yet")

In [ ]:
# [4] TABLE 2 — EDITABILITY. Only interpretable where Table 1 shows working probes.
#     Each editor's BEST arm; read EI against the run's unedited EI, WITH its guard:
#     discworld g = fidelity ratio (>1 destructive); Othello g = li-error-vs-pre
#     (LOW = the model forgot the pre-edit world, i.e. destroyed, not steered).
hdr = (f"{'run':<16}{'env':<11}{'arch':<21}{'uned EI':>9}"
       f"{'PI':>8}{'g':>7}{'ND':>8}{'g':>7}{'GS':>8}{'g':>7}")
print("TABLE 2 — EDITABILITY (best arm per workhorse editor)")
for topic in sorted({r['topic'] for r in rows}):
    print(f"\n## {topic}\n{hdr}\n{'-' * len(hdr)}")
    for r in sorted((r for r in rows if r['topic'] == topic),
                    key=lambda r: (r['env'], r['run'])):
        print(f"{r['run']:<16}{r['env']:<11}{r['arch']:<21}{f(r['unedited_EI']):>9}"
              f"{f(r['PI_EI']):>8}{f(r['PI_guard'], '.2f'):>7}"
              f"{f(r['ND_EI']):>8}{f(r['ND_guard'], '.2f'):>7}"
              f"{f(r['GS_EI']):>8}{f(r['GS_guard'], '.2f'):>7}")

# save the flat rows for downstream views
out = REPO / "notebooks" / "master_table.json"
out.write_text(json.dumps(rows, indent=1, default=float))
print(f"\nwrote {out.relative_to(REPO)}")